In [1]:
source(here::here('MG3.functions1.R'))

# Get genomic coordinates for the key species CDS (Needed to build association rules)

## Using GTFs dowloaded from ENSEMBL FTP.

In [2]:
# run ONCE

GTFdir=paste0(StaticInpDir,"keySpecies/")
columnsToExport=c("seqnames", "start", "end", "width", "strand", "source", "type", "gene_id", "gene_name", "protein_id")
if ("keySpeciesGenomicCoords" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE keySpeciesGenomicCoords')

for (sp in c("hsapiens", relevantSpecies)){
    message(sp)
    if (sp == "hsapiens"){
        fname=KeySpDF$EnsemblGTF[ KeySpDF$HomologueTable == ""]
    }else{
        fname=KeySpDF$EnsemblGTF[ KeySpDF$HomologueTable == sp]
    }
    tmpGTF=import.gff(paste0(GTFdir,fname))
    tmpDT=as.data.table(tmpGTF)
    tmpDT=tmpDT[,..columnsToExport]
    tmpDT$specie=sp
    tmpDT=renameDFcolumn(inDF = tmpDT, incol = "gene_name", innewname = "Name")
    tmpDT=renameDFcolumn(inDF = tmpDT, incol = "gene_id", innewname = "specie_gene_id")
    dbWriteTable(mydbRW, "keySpeciesGenomicCoords", tmpDT, append=TRUE)
}

dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS keySpeciesGenomicCoordsGeneInd ON keySpeciesGenomicCoords(specie_gene_id)')
message("ind1")
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS keySpeciesGenomicCoordsInd ON keySpeciesGenomicCoords(specie, seqnames, start, end, strand)')
message("ind2")
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS keySpeciesGenomicCoordsNameInd ON keySpeciesGenomicCoords(Name, specie)')
message("ind3")
dbExecute(mydbRW, 'CREATE INDEX IF NOT EXISTS keySpeciesGenomicCoordsProteinInd ON keySpeciesGenomicCoords(protein_id)')
message("ind4")

[1] 0

hsapiens

mmusculus

clfamiliaris

fcatus

btaurus

cdromedarius

lafricana

sscrofa

bmusculus



[1] 0

ind1



[1] 0

ind2



[1] 0

ind3



[1] 0

ind4



# Saving transactions to SQL

In [2]:
library(parallel)


In [3]:
outputTableName="keySpBijectiveGenesTransactions5MB"
if (outputTableName %in% dbListTables(mydbRW)) dbExecute(mydbRW, paste('DROP TABLE',outputTableName))


In [4]:
get_specie_transactions=function(sp){
    transactionDTallChrs=NULL
    queryTxt=sprintf('SELECT seqnames, MAX(end) AS expectedLen FROM keySpeciesGenomicCoords WHERE specie="%s" GROUP BY seqnames', sp)
    chrExpLenDF=dbGetQuery(mydbRO, queryTxt)
    chrExpLenDF=chrExpLenDF[ chrExpLenDF$expectedLen > minCHrLim , ]
    
    for ( chrname in chrExpLenDF$seqnames){    
        queryTxt=sprintf('SELECT * FROM keySpeciesGenomicCoords WHERE specie="%s" 
                     AND seqnames="%s" AND type="CDS" AND ( ( specie_gene_id IN 
                     (SELECT DISTINCT ensembl_gene_id FROM HomologyBijectiveWellDefined) )
                    OR 
                     ( specie_gene_id IN  (SELECT DISTINCT homolog FROM HomologyBijectiveWellDefined) ))', sp, chrname) 
        CDS.GR=GRanges(dbGetQuery(mydbRO, queryTxt))
        if (length(CDS.GR)==0) next
        CDS.GR$ROWnumber=1:length(CDS.GR)
        CDS.GRL=split(CDS.GR, CDS.GR$specie_gene_id)
        GENE.SPANS.GRL=endoapply(X = CDS.GRL, FUN = getSpan)
        GENE.SPANS.GR=unlist(GENE.SPANS.GRL)
        GENE.SPANS.GR$specie_gene_id=names(GENE.SPANS.GR)
        GENE.SURROUNDINGS.GR=GENE.SPANS.GR
        start(GENE.SURROUNDINGS.GR)=pmax(start(GENE.SURROUNDINGS.GR)-lookaroundDist, 1)
        end(GENE.SURROUNDINGS.GR)=end(GENE.SURROUNDINGS.GR)+lookaroundDist
        GENE.SURROUNDINGS.HITS=as.data.table(findOverlaps(GENE.SURROUNDINGS.GR, GENE.SURROUNDINGS.GR))
        GENE.SURROUNDINGS.HITS$geneA=GENE.SURROUNDINGS.GR$specie_gene_id[ GENE.SURROUNDINGS.HITS$queryHits]
        GENE.SURROUNDINGS.HITS$geneB=GENE.SURROUNDINGS.GR$specie_gene_id[ GENE.SURROUNDINGS.HITS$subjectHits]
        tmpL=split(GENE.SURROUNDINGS.HITS, GENE.SURROUNDINGS.HITS$queryHits)
        transactionDT=rbindlist(lapply(tmpL, FUN = function(x){ return(data.table(transactionID=unique(x$queryHits), 
                                                                specie_gene_id=c(unique(x$geneA), unique(x$geneB))))}))
        transactionDT=unique(transactionDT)
        transactionDT$specie=sp
        transactionDT$seqnames=chrname
        if (! is.null(transactionDTallChrs)){
            TransactionIDshift=max(transactionDTallChrs$transactionID)
        }else{
            TransactionIDshift=0
        }
        transactionDT$transactionID=transactionDT$transactionID+TransactionIDshift
        transactionDTallChrs=addToDF(transactionDTallChrs, transactionDT)

    }
    
    return(transactionDTallChrs)
}

In [5]:
species_to_do=as.list(c("hsapiens", relevantSpecies))
resulting_DFs <- mclapply(X = species_to_do, FUN = get_specie_transactions, mc.cores = length(species_to_do))


In [6]:
TransactionIDglobalshift=0
for (i in 1:length(resulting_DFs)){
    message("TransactionIDglobalshift=",TransactionIDglobalshift)
    resulting_DFs[[i]]$transactionID=resulting_DFs[[i]]$transactionID + TransactionIDglobalshift
    dbWriteTable(mydbRW, outputTableName, resulting_DFs[[i]], append=TRUE)
    TransactionIDglobalshift=max(resulting_DFs[[i]]$transactionID)
}

TransactionIDglobalshift=0

TransactionIDglobalshift=9754

TransactionIDglobalshift=19508

TransactionIDglobalshift=29261

TransactionIDglobalshift=39014

TransactionIDglobalshift=48766

TransactionIDglobalshift=58517

TransactionIDglobalshift=68092

TransactionIDglobalshift=77823



In [7]:
dbExecute(mydbRW, sprintf('CREATE INDEX IF NOT EXISTS %sIDInd ON %s(transactionID)', outputTableName, outputTableName))
dbExecute(mydbRW, sprintf('CREATE INDEX IF NOT EXISTS %sGeneIDInd ON %s(specie_gene_id)', outputTableName, outputTableName))
dbExecute(mydbRW, sprintf('CREATE INDEX IF NOT EXISTS %sSpecieInd ON %s(specie, seqnames)', outputTableName, outputTableName))


[1] 0

[1] 0

[1] 0

In [8]:
dbListTables(mydbRO)

[1] "BijectiveRelevantSpeciesView"       "GTFs"                              
 [3] "GeneNameSynonims"                   "HMMkeySpecieCrossHits"             
 [5] "HMMkeySpecieCrossHitsCount"         "HomologyBijectiveWellDefined"      
 [7] "RMSK"                               "genomes"                           
 [9] "homology"                           "keyAAsequences"                    
[11] "keySpBijectiveGenesTransactions5MB" "keySpeciesGenomicCoords"           
[13] "relevantSpeciesView"                "seqinfo"                           
[15] "suspectedTEgenes"                   "taxonomy"